# QLoRA Fine-Tuning — Qwen2.5-Coder-7B

**Goal:** test whether targeted *execution-feedback* fine-tuning can reduce the model's
pattern-matching lock-in on adversarially-phrased problems — the failure mode found in
`reports/error_analysis.md` (base model: **54.2% pass@1** on the adversarial eval).

**Run this on a cloud GPU** (Colab free T4 or Kaggle). A 7B model cannot train on a 16GB Mac.

### Techniques encoded here (with sources)
| Choice | Why | Paper |
|---|---|---|
| Grounded, diverse problem families | avoid templated junk | Magicoder / OSS-Instruct [2] |
| Execution-feedback trajectories (wrong attempt → failing test → fix) | learn from test signal, not just final answers | CodeRL [3] |
| Concise structured responses (Approach/Edge/Code/Complexity) | useful to a dev, not a long CoT | Qwen2.5-Coder report [1] |
| QLoRA (4-bit base + LoRA) | fit 7B on a single T4 | QLoRA [6], LoRA [5] |

> **Scope caveat.** The data covers ~27 problem families. Train and eval share *families*
> but never the same inputs (leakage-controlled). Read results as **in-family generalization
> to new inputs**, not broad coding improvement.

*Citations [n] are defined in the **Design rationale & references** section at the bottom.*


## 1. Install dependencies


In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "trl>=0.9" "datasets>=2.20" \
    "accelerate>=0.33" bitsandbytes
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Get the repo and (re)generate the SFT data

The dataset files are gitignored, so we rebuild them from the committed generators — fully
reproducible. Replace the URL with your fork if needed.


In [ ]:
REPO_URL = 'https://github.com/jerrylin-23/Qwencode.git'
import os, subprocess, sys
if not os.path.exists('Qwencode'):
    subprocess.run(['git','clone','--depth','1',REPO_URL], check=True)
%cd Qwencode
!pip -q install -e .
# Rebuild grounded SFT data (solutions + execution-feedback trajectories, leakage-controlled)
!python -m code_model_lab.data.clean_dataset --per_family 60


## 3. Load data and format with the Qwen chat template


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

SYSTEM = ('You are a coding assistant. Read the problem requirements literally and implement '
          'them directly. Do not reuse a similar well-known solution without checking it fits.')

def to_text(ex):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':ex['instruction']},
            {'role':'assistant','content':ex['response']}]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}

ds = load_dataset('json', data_files={
    'train':'data/processed/sft_train.jsonl',
    'valid':'data/processed/sft_valid.jsonl'})
ds = ds.map(to_text, remove_columns=['instruction','response'])
print(ds); print('\n--- one formatted example ---\n', ds['train'][0]['text'][:600])


## 4. Load the base model in 4-bit (QLoRA)


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4: fp16 (use bfloat16 on A100/L4)
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    # all linear projections, not just q/v — better adapter coverage
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 5. Train

Start with **1 epoch** to confirm the loss drops and the pipeline is sound, then raise to 2–3.


In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir='outputs/adapters/sft_v1',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    fp16=True,
    optim='paged_adamw_8bit',
    max_seq_length=1024,
    packing=True,
    dataset_text_field='text',
    report_to='none',
)
trainer = SFTTrainer(model=model, args=cfg,
                     train_dataset=ds['train'], eval_dataset=ds['valid'])
trainer.train()
trainer.save_model('outputs/adapters/sft_v1')
tok.save_pretrained('outputs/adapters/sft_v1')


## 6. Qualitative check on a *trap* prompt

`climb_no_double2` is the family the base model solves as plain Fibonacci even after retries.
Does the adapter read the constraint now?


In [ ]:
PROMPT = ('Problem:\nYou climb n stairs taking 1 or 2 steps at a time, but you may NEVER '
          'take two 2-steps in a row. Return the number of distinct ways to reach the top.\n\n'
          'Starter Code:\ndef climb_no_double2(n):\n    pass')
msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':PROMPT}]
inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
out = model.generate(inputs, max_new_tokens=400, do_sample=False)
print(tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
# correct answers: n=5 -> 9, n=10 -> 64 (NOT Fibonacci 13 / 144)


## 7. Quantitative eval — run the committed adversarial harness against the adapter

This is the headline number. The repo's HF client supports an adapter path; point the eval
config at it and compare pass@1 to the base model's **54.2%**.


In [ ]:
# Option A (in-notebook, GPU): merge or load adapter via the repo's HFClient.
# Edit configs/model.yaml -> model_type: hf, adapter_path: outputs/adapters/sft_v1
import yaml, pathlib
mc = yaml.safe_load(open('configs/model.yaml'))
mc.update({'model_type':'hf','model_name':MODEL_ID,'adapter_path':'outputs/adapters/sft_v1','load_in_4bit':True})
yaml.safe_dump(mc, open('configs/model_hf_adapter.yaml','w'))

# point the adversarial eval config at the HF adapter model
ec = yaml.safe_load(open('configs/eval_adversarial_pass1.yaml'))
ec['model_config'] = 'configs/model_hf_adapter.yaml'
yaml.safe_dump(ec, open('configs/eval_adv_adapter.yaml','w'))

import os; os.environ['EVAL_MODE']='real'
!python -m code_model_lab.eval.run_algorithm_eval --config configs/eval_adv_adapter.yaml
# Compare the printed pass@1 to the base baseline (54.2%). A real win moves this up.


## 8. Next: the ablation

To show *which* technique helped, train a second adapter on **solution records only** (drop the
feedback trajectories) and eval it the same way. The plan's ablation table:

| Variant | adversarial pass@1 |
|---|---|
| Base (no fine-tune) | 54.2% |
| SFT, solutions only | ? |
| SFT + execution-feedback | ? |

If the feedback adapter beats solutions-only, that isolates CodeRL-style execution feedback as
the cause — the project's headline result.


## Design rationale & references

Every non-obvious choice in this notebook traces to a specific result in the literature.

### Data design
| Decision | Rationale | Source |
|---|---|---|
| Generate data from real algorithmic *problem families* rather than free-form LLM prompts | Grounding synthetic instruction data in concrete code/seeds yields more diverse, less hallucinated, higher-quality training data than ungrounded self-instruct | Wei et al., *Magicoder / OSS-Instruct* [2]; Wang et al., *Self-Instruct* [8] |
| Heavy dedup + leakage control + difficulty/family balancing | Data cleaning and balanced mixing are the dominant levers behind Qwen2.5-Coder's gains; contamination control is required for valid eval | Hui et al., *Qwen2.5-Coder Technical Report* [1] |
| **Execution-feedback trajectories** (wrong attempt → failing test → corrected code) | Learning from unit-test execution signals — not just gold final answers — is what improves program synthesis; this is the core of CodeRL | Le et al., *CodeRL* [3] |
| Framing the fix as *re-read the requirement, then correct* | LLMs can repair their own code when shown the failing execution result; self-debugging from test feedback is an established, effective recipe | Chen et al., *Teaching LLMs to Self-Debug* [4] |
| Concise, structured responses (Approach/Edge/Code/Complexity) instead of long chain-of-thought | Keep completions actionable for a developer; the project targets useful output, not a long hidden reasoning trace | Qwen2.5-Coder report [1] (data-style guidance) |

### Training design
| Decision | Rationale | Source |
|---|---|---|
| **LoRA** (freeze base, train low-rank adapters) | Matches full fine-tuning quality at a fraction of trainable params; enables single-GPU 7B tuning | Hu et al., *LoRA* [5] |
| **4-bit NF4 + double quantization** base | NF4 is information-theoretically suited to normally-distributed weights; double-quant cuts memory further with negligible quality loss — lets a 7B fit a 16GB T4 | Dettmers et al., *QLoRA* [6] |
| LoRA on **all linear projections** (q,k,v,o,gate,up,down), not just q/v | QLoRA found adapter coverage of *all* linear layers is needed to match full-FT performance; restricting to q/v underfits | Dettmers et al., *QLoRA* [6] |
| **Paged 8-bit AdamW** optimizer | 8-bit block-wise optimizer states cut memory ~4×; paging avoids OOM spikes on long sequences | Dettmers et al., *8-bit Optimizers* [7]; *QLoRA* [6] |
| Apply the model's **chat template** for SFT formatting | Training in the same instruction/chat format used at inference is standard for instruction tuning and avoids template mismatch | Ouyang et al., *InstructGPT* [9] |

### Evaluation design (why we measure on the adversarial set, not canonical pass@1)
| Decision | Rationale | Source |
|---|---|---|
| Headline metric = pass@1 on *adversarially-phrased* problems | Canonical/algorithmic benchmarks saturate for a strong 7B coder (we measured ~100%); a practical, instruction-sensitive eval exposes real gaps | Zhuo et al., *BigCodeBench* [10] (practical, instruction-following code eval) |
| Report base-vs-adapter and an ablation (solutions-only vs +feedback) | Improvement claims need a held-out comparison that isolates the cause, not just a single after-number | Standard ablation practice; CodeRL [3] compares with/without execution feedback |

### References
1. Hui et al. (2024). *Qwen2.5-Coder Technical Report.* arXiv:2409.12186 — https://arxiv.org/abs/2409.12186
2. Wei et al. (2023). *Magicoder: Source Code Is All You Need* (OSS-Instruct). arXiv:2312.02120 — https://arxiv.org/abs/2312.02120
3. Le et al. (2022). *CodeRL: Mastering Code Generation through Pretrained Models and Deep Reinforcement Learning.* arXiv:2207.01780 — https://arxiv.org/abs/2207.01780
4. Chen et al. (2023). *Teaching Large Language Models to Self-Debug.* arXiv:2304.05128 — https://arxiv.org/abs/2304.05128
5. Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.* arXiv:2106.09685 — https://arxiv.org/abs/2106.09685
6. Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.* arXiv:2305.14314 — https://arxiv.org/abs/2305.14314
7. Dettmers et al. (2021). *8-bit Optimizers via Block-wise Quantization.* arXiv:2110.02861 — https://arxiv.org/abs/2110.02861
8. Wang et al. (2022). *Self-Instruct: Aligning Language Models with Self-Generated Instructions.* arXiv:2212.10560 — https://arxiv.org/abs/2212.10560
9. Ouyang et al. (2022). *Training language models to follow instructions with human feedback* (InstructGPT). arXiv:2203.02155 — https://arxiv.org/abs/2203.02155
10. Zhuo et al. (2024). *BigCodeBench: Benchmarking Code Generation with Diverse Function Calls and Complex Instructions.* arXiv:2406.15877 — https://arxiv.org/abs/2406.15877

*Note on scope:* references justify the **methodology**. The numeric results in this notebook
come only from runs against the project's own held-out evals — no external numbers are claimed.
